# A custom model: software training to CIM inference
Define a three-layer MLP, train it in PyTorch, save its original parameters, rebuild it, and compare digital, ideal-device CIM and noisy-device CIM accuracy. Run cells in order in an environment with MemFast and torchvision installed. GPU is used when available; CPU also works.

In [ ]:
import os
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from memfast import Config, Nonidealities, Simulator, convert_model

torch.manual_seed(135)
torch.set_num_threads(4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
data_root = Path(os.environ.get("MEMFAST_MNIST_ROOT", "data"))
output = Path("output/custom_mlp")
output.mkdir(parents=True, exist_ok=True)
checkpoint = output / "mnist.pt"
hidden_sizes = (128, 64)
epochs = 5
batch_size = 128
print("Device:", device)

## 1. Define the network and prepare data
Change the dimensions here for your own input and classes. The checkpoint must be restored into the same architecture.

In [ ]:
class CustomMLP(nn.Module):
    def __init__(self, input_features=784, hidden=(128, 64), classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_features, hidden[0])
        self.fc2 = nn.Linear(hidden[0], hidden[1])
        self.fc3 = nn.Linear(hidden[1], classes)

    def forward(self, x):
        x = torch.relu(self.fc1(self.flatten(x)))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
training = datasets.MNIST(data_root, train=True, download=True, transform=transform)
testing = datasets.MNIST(data_root, train=False, download=True, transform=transform)
train_loader = DataLoader(training, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testing, batch_size=batch_size)
model = CustomMLP(hidden=hidden_sizes).float().to(device)
model

## 2. Train entirely in software and save the checkpoint
No simulator participates in this stage. Cross entropy receives raw class logits.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(epochs):
    model.train()
    loss_sum = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(model(images), labels)
        loss.backward()
        optimizer.step()
        loss_sum += float(loss.detach()) * labels.numel()
    print(f"Epoch {epoch + 1}: training loss {loss_sum / len(training):.4f}")
torch.save(
    {
        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "hidden_sizes": hidden_sizes,
        "seed": 135,
    },
    checkpoint,
)
print("Saved:", checkpoint)

## 3. Rebuild and load, then evaluate the digital baseline
This is also the entry point for a checkpoint you trained elsewhere.

In [ ]:
payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
restored = CustomMLP(hidden=tuple(payload["hidden_sizes"]))
restored.load_state_dict(payload["state_dict"], strict=True)
restored = restored.to(device).eval()


@torch.inference_mode()
def accuracy(network):
    correct = count = 0
    network.eval()
    for images, labels in test_loader:
        logits = network(images.to(device))
        assert torch.isfinite(logits).all()
        correct += int((logits.argmax(1).cpu() == labels).sum())
        count += labels.numel()
    return correct / count, count


digital_accuracy, count = accuracy(restored)
print(f"Digital: {digital_accuracy:.2%}, {count} test images")

## 4. Compare ideal and noisy CIM inference
Both CIM runs use A8W8 INT, one-bit slices, 64x64 arrays and ADC6. Ideal devices still include quantization and ADC conversion. Only programming variation differs; its fixed realization is identified by the seed and layer names.

In [ ]:
scores = {"digital": digital_accuracy}
for name, write_sigma in [("ideal CIM", 0.0), ("write-noisy CIM", 0.02)]:
    config = Config(nonidealities=Nonidealities(write_variation=write_sigma), seed=135)
    engine = Simulator(config, device=device)
    analog = convert_model(restored, engine)  # Default: all supported layers.
    scores[name], samples = accuracy(analog)
    print(f"{name}: {scores[name]:.2%}, {samples} test images")
    del analog, engine

# Map the first two layers only; fc3 remains a software classifier.
engine = Simulator(Config(), device=device)
partial = convert_model(restored, engine, include=("fc1", "fc2"))
scores["partial noisy CIM"], _ = accuracy(partial)
print(f"Partial noisy CIM: {scores['partial noisy CIM']:.2%}")
assert isinstance(partial.fc3, nn.Linear)
scores

The numbered MLP script offers the same training/checkpoint workflow for command-line runs and a HAT stage. HAT uses analog forward values and FP32 surrogate backward; it starts from the digital checkpoint.